# NB12 — Top-2 confusion matrices and learning curves

**Project:** Comparative Evaluation of Dimensionality Reduction, Machine Learning, and Deep Tabular Models for Morphometric Classification of INIAP Dry Bean Cultivars

This notebook regenerates two publication figures for the manuscript:

- **Figure 4A–B:** mean row-normalized outer-test OOF confusion matrices for **TabPFN** and **LDA-XGBoost**.
- **Figure 8A–B:** leakage-safe learning curves for **TabPFN** and **LDA-XGBoost**.

## Methodological principles

1. Confusion matrices use only saved **outer-test OOF predictions** from `FULL_11`.
2. Repeated seeds are not pooled as independent grains. A row-normalized confusion matrix is computed separately for each seed and then averaged across the three seeds.
3. The existing LDA-XGBoost learning-curve results from NB07 are reused.
4. TabPFN learning curves are newly computed using the **same seeds, outer folds, and training fractions** as NB07.
5. Only the outer-training portion is subsampled. The outer-test fold is never used for preprocessing, parameter selection, or training.
6. TabPFN is treated as the fixed pretrained comparator used in NB08; no fine-tuning or inner-CV hyperparameter search is introduced here.
7. Checkpoints are written after every completed TabPFN learning-curve fit.


## NB12 FIXED V2 — correction note

This revision corrects the failure observed in Section 6. The previous Colab run contained a truncated/non-ASCII API key (`…`) in `TABPFN_TOKEN`, which caused an HTTP-header encoding error and then `TabPFNLicenseError`. V2 validates the token before fitting, prompts securely for a replacement when needed, retries the interrupted fit once, writes an atomic checkpoint after every successful fit, and clears GPU memory between fits. Figure 4 retains the automatic white-on-dark / black-on-light annotation logic.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Project paths and configuration


In [ ]:
from pathlib import Path
import os, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATA = ROOT / '01_DATA'
RESULTS = ROOT / '03_RESULTS'
FIGURES = ROOT / '05_FIGURES'
LOGS = ROOT / '07_LOGS'

DATASET = DATA / 'INIAP_Dataset.xlsx'
OOF_FILE = RESULTS / 'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE' / 'all_oof_harmonized.csv'
LC_EXISTING = RESULTS / 'NB07_ROC_LEARNING' / 'learning_curve_by_fold.csv'

OUT = RESULTS / 'NB12_TOP2_DIAGNOSTICS'
FIG = FIGURES / 'NB12_TOP2_DIAGNOSTICS'
LOG = LOGS / 'NB12_TOP2_DIAGNOSTICS'
for p in [OUT, FIG, LOG]:
    p.mkdir(parents=True, exist_ok=True)

SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
TRAIN_FRACTIONS = [0.20, 0.40, 0.60, 0.80, 1.00]
TARGET = 'Class'
CLASSES = ['INIAP 420', 'INIAP 425', 'INIAP 481', 'INIAP 485']

for p in [DATASET, OOF_FILE, LC_EXISTING]:
    assert p.exists(), f'Missing required file: {p}'

print('ROOT:', ROOT)
print('OOF predictions:', OOF_FILE)
print('Existing learning curves:', LC_EXISTING)
print('NB12 results:', OUT)
print('NB12 figures:', FIG)


## 2. Imports


In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    DEVICE = 'cpu'

print('DEVICE:', DEVICE)
if DEVICE != 'cuda':
    print('NOTE: Figure 4 is fine on CPU. GPU is strongly recommended for the TabPFN learning-curve section.')


## 3. Figure 4A–B — TabPFN and LDA-XGBoost confusion matrices

For each seed, all five outer-test folds are concatenated to recover one OOF prediction per grain.
The confusion matrix is row-normalized within that seed. The published matrix is the arithmetic mean
of the three seed-specific row-normalized matrices.

This avoids incorrectly treating the same physical grain from three repeated seeds as three independent observations.


In [ ]:
import matplotlib.patheffects as pe

oof = pd.read_csv(OOF_FILE)

required = {
    'row_id',
    'y_true',
    'y_pred',
    'model',
    'seed',
    'outer_fold',
    'feature_set'
}

assert required.issubset(oof.columns), required - set(oof.columns)

models = ['TabPFN', 'LDA_XGBoost']
panel_letters = ['A', 'B']

# ============================================================
# 1. Validate OOF structure
# ============================================================

for model in models:
    g = oof[
        (oof['feature_set'] == 'FULL_11') &
        (oof['model'] == model)
    ].copy()

    assert len(g) == 12000, (model, len(g))

    for seed in SEEDS:
        gs = g[g['seed'] == seed]

        assert len(gs) == 4000, (model, seed, len(gs))
        assert gs['row_id'].nunique() == 4000, (
            model,
            seed,
            'row_id duplication'
        )

# ============================================================
# 2. Compute seed-specific normalized confusion matrices
# ============================================================

seed_matrices = {}

for model in models:

    mats = []

    g = oof[
        (oof['feature_set'] == 'FULL_11') &
        (oof['model'] == model)
    ].copy()

    for seed in SEEDS:

        gs = g[g['seed'] == seed]

        cm = confusion_matrix(
            gs['y_true'],
            gs['y_pred'],
            labels=CLASSES,
            normalize='true'
        )

        mats.append(cm)

    seed_matrices[model] = np.stack(
        mats,
        axis=0
    )

# Mean normalized confusion matrix across the three seeds
mean_matrices = {
    model: seed_matrices[model].mean(axis=0)
    for model in models
}

# ============================================================
# 3. Publication-quality Figure 4
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.6, 5.2),
    constrained_layout=True
)

images = []

for ax, model, letter in zip(
    axes,
    models,
    panel_letters
):

    mat = mean_matrices[model]

    # Heatmap
    im = ax.imshow(
        mat,
        vmin=0,
        vmax=1,
        cmap='Blues'
    )

    images.append(im)

    # Axis ticks
    ax.set_xticks(
        range(len(CLASSES))
    )

    ax.set_yticks(
        range(len(CLASSES))
    )

    ax.set_xticklabels(
        CLASSES,
        rotation=35,
        ha='right',
        fontsize=10
    )

    ax.set_yticklabels(
        CLASSES,
        fontsize=10
    )

    # Axis labels
    ax.set_xlabel(
        'Predicted cultivar',
        fontsize=11
    )

    ax.set_ylabel(
        'True cultivar',
        fontsize=11
    )

    # ========================================================
    # Panel letter A / B
    # WHITE text + black outline
    # ========================================================

    ax.text(
        0.02,
        0.98,
        letter,
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=15,
        fontweight='bold',
        color='white',
        path_effects=[
            pe.withStroke(
                linewidth=2.5,
                foreground='black'
            )
        ]
    )

    # ========================================================
    # Cell values
    # White on dark cells / black on light cells
    # ========================================================

    for i in range(mat.shape[0]):

        for j in range(mat.shape[1]):

            value = mat[i, j]

            text_color = (
                'white'
                if value >= 0.50
                else 'black'
            )

            ax.text(
                j,
                i,
                f'{value:.3f}',
                ha='center',
                va='center',
                fontsize=10,
                fontweight='bold',
                color=text_color
            )

# ============================================================
# 4. Shared color bar
# ============================================================

cbar = fig.colorbar(
    images[0],
    ax=axes,
    fraction=0.025,
    pad=0.03
)

cbar.set_label(
    'Mean row-normalized proportion',
    fontsize=10
)

cbar.ax.tick_params(
    labelsize=9
)

# ============================================================
# 5. Output files
# ============================================================

fig4_png = (
    FIG /
    'Figure4_Confusion_Matrices_TabPFN_LDAXGBoost_PUBLICATION.png'
)

fig4_pdf = (
    FIG /
    'Figure4_Confusion_Matrices_TabPFN_LDAXGBoost_PUBLICATION.pdf'
)

fig4_svg = (
    FIG /
    'Figure4_Confusion_Matrices_TabPFN_LDAXGBoost_PUBLICATION.svg'
)

# ============================================================
# 6. Save at publication quality
# ============================================================

fig.savefig(
    fig4_png,
    dpi=600,
    bbox_inches='tight'
)

fig.savefig(
    fig4_pdf,
    bbox_inches='tight'
)

fig.savefig(
    fig4_svg,
    bbox_inches='tight'
)

plt.show()

print('Saved:')
print(fig4_png)
print(fig4_pdf)
print(fig4_svg)


### Figure 4 caption for the manuscript

**Figure 4.** Mean row-normalized outer-test out-of-fold confusion matrices across the three repeated seeds for the two leading models using the full 11-descriptor representation: **(A)** TabPFN and **(B)** LDA-XGBoost. For each seed, every grain contributed exactly one outer-test prediction; seed-specific normalized matrices were then averaged, so repeated seeds were not treated as independent grains.


## 4. Prepare the TabPFN learning-curve analysis

The existing NB07 file already contains LDA-XGBoost learning curves for the exact design:
three seeds × five outer folds × five training fractions.

TabPFN is not present in NB07 and therefore must be evaluated separately.


In [ ]:
# Install only if needed.
try:
    from tabpfn import TabPFNClassifier
    print('TabPFN already available.')
except Exception:
    print('Installing TabPFN...')
    !pip -q install tabpfn
    from tabpfn import TabPFNClassifier

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('TabPFN device:', DEVICE)

if DEVICE != 'cuda':
    print('WARNING: Use a GPU runtime for this section if possible.')


### Authentication note

The study uses local TabPFN inference in the Colab runtime. Depending on the installed TabPFN release
and whether the licensed model checkpoint is already cached, the package may request a Prior Labs/Hugging Face
credential to access the pretrained checkpoint. The credential is used only for model access and is not written to Drive.

If your current environment already has access to the checkpoint, no prompt may be required.


In [ ]:
from getpass import getpass
import os
from pathlib import Path

# ============================================================
# TABPFN AUTHENTICATION — SAFE AND VALIDATED
# ============================================================
# IMPORTANT:
# - Paste the COMPLETE Prior Labs API key, not the masked value shown
#   as dots/ellipsis in a previous output.
# - The key stays only in RAM. It is never written to Drive.

TABPFN_CACHE_FILES = [
    Path.home() / '.cache' / 'tabpfn' / 'auth_token',
    Path.home() / '.tabpfn' / 'token',
]

def token_looks_complete(token):
    if token is None:
        return False
    token = token.strip()
    if not token:
        return False
    # The error in the previous run came from a Unicode ellipsis (…).
    if '…' in token or '\u2026' in token:
        return False
    # API keys used in HTTP headers must be ASCII.
    if not token.isascii():
        return False
    # Reject obvious placeholders/truncated displays.
    lowered = token.lower()
    if 'your-api-key' in lowered or '<' in token or '>' in token:
        return False
    if len(token) < 20:
        return False
    return True

def clear_bad_tabpfn_token():
    os.environ.pop('TABPFN_TOKEN', None)
    for p in TABPFN_CACHE_FILES:
        try:
            if p.exists():
                p.unlink()
        except Exception:
            pass

def request_tabpfn_token(force=False):
    token = '' if force else os.environ.get('TABPFN_TOKEN', '').strip()

    if not token_looks_complete(token):
        if token:
            print('⚠️ The TABPFN_TOKEN currently in memory is incomplete or contains invalid characters.')
            print('It will be discarded before continuing.')
        clear_bad_tabpfn_token()

        print('Open your Prior Labs account and copy the COMPLETE API key.')
        print('Do not copy dots, bullets, or the character “…”.')
        token = getpass('Paste COMPLETE TABPFN_TOKEN (hidden; not saved): ').strip()

        if not token_looks_complete(token):
            clear_bad_tabpfn_token()
            raise RuntimeError(
                'The pasted TABPFN_TOKEN is still incomplete/invalid. '
                'Copy the complete API key from your Prior Labs account and rerun this cell.'
            )

        os.environ['TABPFN_TOKEN'] = token

    print('✅ TABPFN_TOKEN validated locally and loaded in RAM only.')
    return token

_ = request_tabpfn_token()


## 6. Compute only the missing TabPFN learning curves

There are 75 TabPFN fits in total:

- 3 seeds
- 5 outer folds
- 5 training fractions

A checkpoint is saved after every completed fit. Rerunning the cell resumes from the existing checkpoint.


## 6. Compute only the missing TabPFN learning curves

There are 75 TabPFN fits in total:

- 3 seeds
- 5 outer folds
- 5 training fractions

A checkpoint is saved after every completed fit. Rerunning the cell resumes from the existing checkpoint.


## 5. Load the dataset and reproduce the NB07 outer folds


In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation': 'AspectRatio'})

FEATURES_11 = [
    'Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio',
    'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness'
]

assert len(df) == 4000
assert TARGET in df.columns
assert set(FEATURES_11).issubset(df.columns)

X = df[FEATURES_11].astype(np.float32)

le = LabelEncoder()
y = pd.Series(le.fit_transform(df[TARGET].astype(str)), index=df.index)

assert list(le.classes_) == CLASSES, list(le.classes_)

def stratified_fraction_indices(y_arr, frac, seed):
    if frac >= 0.999999:
        return np.arange(len(y_arr))

    idx = np.arange(len(y_arr))
    keep, _ = train_test_split(
        idx,
        train_size=frac,
        stratify=y_arr,
        random_state=seed
    )
    return np.sort(keep)

print('Dataset loaded:', X.shape)
print('Classes:', list(le.classes_))


## 6. Compute only the missing TabPFN learning curves

There are **75 TabPFN fits** in total (3 seeds × 5 outer folds × 5 training fractions).

This corrected implementation:

- validates the Prior Labs API key before starting;
- detects truncated/non-ASCII tokens such as the `…` value that caused the previous failure;
- automatically prompts for a fresh key if a TabPFN license/authentication error occurs;
- saves an atomic checkpoint after **every successful fit**;
- resumes only the missing seed–fold–fraction combinations;
- clears GPU memory after every fit to reduce cumulative CUDA-memory problems.


In [ ]:
# Section 6 is self-contained: authentication helpers are defined here too.
from getpass import getpass
import os
from pathlib import Path

TABPFN_CACHE_FILES = [
    Path.home() / '.cache' / 'tabpfn' / 'auth_token',
    Path.home() / '.tabpfn' / 'token',
]

def token_looks_complete(token):
    if token is None:
        return False
    token = token.strip()
    if not token:
        return False
    if '…' in token or '\\u2026' in token:
        return False
    if not token.isascii():
        return False
    lowered = token.lower()
    if 'your-api-key' in lowered or '<' in token or '>' in token:
        return False
    if len(token) < 20:
        return False
    return True

def clear_bad_tabpfn_token():
    os.environ.pop('TABPFN_TOKEN', None)
    for p in TABPFN_CACHE_FILES:
        try:
            if p.exists():
                p.unlink()
        except Exception:
            pass

def request_tabpfn_token(force=False):
    token = '' if force else os.environ.get('TABPFN_TOKEN', '').strip()
    if not token_looks_complete(token):
        if token:
            print('⚠️ The TABPFN_TOKEN in memory is incomplete or contains invalid characters.')
        clear_bad_tabpfn_token()
        print('Copy the COMPLETE API key from your Prior Labs account.')
        print('Do not copy dots, bullets, or the character “…”.')
        token = getpass('Paste COMPLETE TABPFN_TOKEN (hidden; not saved): ').strip()
        if not token_looks_complete(token):
            clear_bad_tabpfn_token()
            raise RuntimeError(
                'The pasted TABPFN_TOKEN is incomplete/invalid. '
                'Copy the complete API key from Prior Labs and rerun Section 6.'
            )
        os.environ['TABPFN_TOKEN'] = token
    print('✅ TABPFN_TOKEN validated locally and loaded in RAM only.')
    return token

# ============================================================
# SECTION 6 — ROBUST TABPFN LEARNING CURVES WITH RESUME
# ============================================================
import gc
import traceback
from pathlib import Path

try:
    from tabpfn.errors import TabPFNLicenseError
except Exception:
    class TabPFNLicenseError(Exception):
        pass

checkpoint = OUT / 'tabpfn_learning_curve_CHECKPOINT.csv'
final_file = OUT / 'tabpfn_learning_curve_by_fold.csv'
failure_log = LOG / 'NB12_tabpfn_learning_curve_failure.txt'

LC_COLUMNS = [
    'model', 'seed', 'outer_fold', 'train_fraction', 'n_train',
    'train_macro_f1', 'test_macro_f1', 'fit_predict_seconds'
]

EXPECTED_KEYS = {
    (int(seed), int(fold), round(float(frac), 2))
    for seed in SEEDS
    for fold in range(1, OUTER_FOLDS + 1)
    for frac in TRAIN_FRACTIONS
}
EXPECTED_ROWS = len(EXPECTED_KEYS)


def atomic_save_csv(frame, path):
    """Write checkpoint safely so an interrupted write does not corrupt progress."""
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    tmp.replace(path)


def normalize_checkpoint(frame):
    """Validate/clean a previous checkpoint and keep only valid NB12 rows."""
    if frame is None or frame.empty:
        return pd.DataFrame(columns=LC_COLUMNS)

    missing = [c for c in LC_COLUMNS if c not in frame.columns]
    if missing:
        raise ValueError(f'Checkpoint is missing columns: {missing}')

    x = frame[LC_COLUMNS].copy()
    x['seed'] = pd.to_numeric(x['seed'], errors='coerce')
    x['outer_fold'] = pd.to_numeric(x['outer_fold'], errors='coerce')
    x['train_fraction'] = pd.to_numeric(x['train_fraction'], errors='coerce').round(2)

    x = x[
        (x['model'].astype(str) == 'TabPFN') &
        x['seed'].isin(SEEDS) &
        x['outer_fold'].isin(range(1, OUTER_FOLDS + 1)) &
        x['train_fraction'].isin([round(float(f), 2) for f in TRAIN_FRACTIONS])
    ].copy()

    x = x.dropna(subset=['seed', 'outer_fold', 'train_fraction'])
    x['seed'] = x['seed'].astype(int)
    x['outer_fold'] = x['outer_fold'].astype(int)

    x = (
        x.drop_duplicates(['seed', 'outer_fold', 'train_fraction'], keep='last')
         .sort_values(['seed', 'outer_fold', 'train_fraction'])
         .reset_index(drop=True)
    )
    return x


def completed_keys(frame):
    if frame.empty:
        return set()
    return set(zip(
        frame['seed'].astype(int),
        frame['outer_fold'].astype(int),
        frame['train_fraction'].astype(float).round(2)
    ))


def load_progress():
    # Prefer the completed file only if it contains the full 75 valid rows.
    candidates = [checkpoint, final_file]
    for path in candidates:
        if not path.exists():
            continue
        try:
            frame = normalize_checkpoint(pd.read_csv(path))
            if len(frame):
                print(f'Loaded {len(frame)}/{EXPECTED_ROWS} completed fits from: {path.name}')
                return frame
        except Exception as e:
            print(f'⚠️ Ignoring invalid checkpoint {path.name}: {e}')
    return pd.DataFrame(columns=LC_COLUMNS)


def clear_gpu():
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def build_tabpfn_classifier():
    # Keep the same TabPFN configuration used in NB08.
    try:
        return TabPFNClassifier(device=DEVICE)
    except TypeError:
        return TabPFNClassifier()


def run_one_fit(Xsub, ysub, Xte, yte, seed, fold, frac):
    """Run one fit; if auth fails, request a fresh token and retry exactly once."""
    last_error = None

    for attempt in (1, 2):
        clf = None
        try:
            # Make sure an obviously truncated token cannot reach the HTTP layer.
            request_tabpfn_token(force=False)

            clf = build_tabpfn_classifier()
            t0 = time.time()
            clf.fit(Xsub, ysub)
            train_pred = clf.predict(Xsub)
            test_pred = clf.predict(Xte)
            elapsed = time.time() - t0

            return {
                'model': 'TabPFN',
                'seed': int(seed),
                'outer_fold': int(fold),
                'train_fraction': float(frac),
                'n_train': int(len(ysub)),
                'train_macro_f1': float(f1_score(ysub, train_pred, average='macro')),
                'test_macro_f1': float(f1_score(yte, test_pred, average='macro')),
                'fit_predict_seconds': float(elapsed),
            }

        except TabPFNLicenseError as e:
            last_error = e
            print('\n❌ TabPFN authentication/license error.')
            print(str(e))

            if attempt == 1:
                print('\nThe stored key will be cleared. Paste the COMPLETE Prior Labs API key again.')
                clear_bad_tabpfn_token()
                request_tabpfn_token(force=True)
                print('Retrying the same fit once...\n')
                continue
            raise

        except UnicodeEncodeError as e:
            # Direct safeguard for the exact previous error (Unicode ellipsis in token).
            last_error = e
            if attempt == 1:
                print('\n❌ The API key contains a non-ASCII/truncated character (for example “…").')
                clear_bad_tabpfn_token()
                request_tabpfn_token(force=True)
                print('Retrying the same fit once...\n')
                continue
            raise

        finally:
            if clf is not None:
                del clf
            clear_gpu()

    raise RuntimeError(f'TabPFN fit failed after reauthentication: {last_error}')


# ------------------------------------------------------------------
# Load prior progress and run only missing combinations.
# ------------------------------------------------------------------
tab_lc = load_progress()
done = completed_keys(tab_lc)

print(f'Progress before run: {len(done)}/{EXPECTED_ROWS} fits complete.')
print(f'Remaining: {EXPECTED_ROWS - len(done)} fits.')

try:
    for seed in SEEDS:
        outer = StratifiedKFold(
            n_splits=OUTER_FOLDS,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, te) in enumerate(outer.split(X, y), start=1):
            Xtr_full = X.iloc[tr].to_numpy(np.float32)
            ytr_full = y.iloc[tr].to_numpy()
            Xte = X.iloc[te].to_numpy(np.float32)
            yte = y.iloc[te].to_numpy()

            for frac in TRAIN_FRACTIONS:
                frac_key = round(float(frac), 2)
                key = (int(seed), int(fold), frac_key)

                if key in done:
                    print('SKIP completed:', key)
                    continue

                sub_local = stratified_fraction_indices(
                    ytr_full,
                    frac,
                    seed=seed * 1000 + fold * 10 + int(round(frac * 100))
                )
                Xsub = Xtr_full[sub_local]
                ysub = ytr_full[sub_local]

                print(
                    f'RUN TabPFN | seed={seed} | fold={fold}/{OUTER_FOLDS} | '
                    f'fraction={frac:.2f} | n_train={len(ysub)} | '
                    f'progress={len(done)+1}/{EXPECTED_ROWS}',
                    flush=True
                )

                row = run_one_fit(
                    Xsub=Xsub,
                    ysub=ysub,
                    Xte=Xte,
                    yte=yte,
                    seed=seed,
                    fold=fold,
                    frac=frac
                )

                tab_lc = pd.concat([tab_lc, pd.DataFrame([row])], ignore_index=True)
                tab_lc = normalize_checkpoint(tab_lc)

                # Save after EVERY successful fit.
                atomic_save_csv(tab_lc, checkpoint)
                done = completed_keys(tab_lc)

                print(
                    f'✅ Saved checkpoint: {len(done)}/{EXPECTED_ROWS} completed | '
                    f'test macro-F1={row["test_macro_f1"]:.4f}'
                )

    # Final integrity checks.
    done = completed_keys(tab_lc)
    missing_keys = EXPECTED_KEYS - done

    print('\nCompleted rows:', len(tab_lc), 'Expected:', EXPECTED_ROWS)

    if missing_keys:
        raise RuntimeError(
            f'TabPFN learning curve is incomplete: {len(missing_keys)} fits are still missing. '
            'Rerun this cell; it will continue from the checkpoint.'
        )

    assert len(tab_lc) == EXPECTED_ROWS
    atomic_save_csv(tab_lc, final_file)

    if failure_log.exists():
        failure_log.unlink()

    print('\n✅ TabPFN learning curves completed successfully.')
    print('Final file:', final_file)

except Exception as e:
    # Progress already completed remains safely stored in checkpoint.
    failure_log.write_text(
        'NB12 TabPFN learning-curve failure\n\n'
        + repr(e)
        + '\n\n'
        + traceback.format_exc(),
        encoding='utf-8'
    )
    print('\n❌ The run stopped, but completed fits are preserved in the checkpoint.')
    print('Failure log:', failure_log)
    print('Fix the reported issue and rerun this same cell to resume.')
    raise


## 7. Figure 8A–B — TabPFN and LDA-XGBoost learning curves


In [ ]:
existing_lc = pd.read_csv(LC_EXISTING)
lda_lc = existing_lc[existing_lc['model'] == 'LDA_XGBoost'].copy()

assert len(lda_lc) == 75, len(lda_lc)
assert len(tab_lc) == 75, len(tab_lc)

combined = pd.concat(
    [
        tab_lc[['model', 'seed', 'outer_fold', 'train_fraction', 'n_train',
                'train_macro_f1', 'test_macro_f1']],
        lda_lc[['model', 'seed', 'outer_fold', 'train_fraction', 'n_train',
                'train_macro_f1', 'test_macro_f1']]
    ],
    ignore_index=True
)

summary = (
    combined
    .groupby(['model', 'train_fraction'], as_index=False)
    .agg(
        mean_n_train=('n_train', 'mean'),
        mean_train_macro_f1=('train_macro_f1', 'mean'),
        sd_train_macro_f1=('train_macro_f1', 'std'),
        mean_test_macro_f1=('test_macro_f1', 'mean'),
        sd_test_macro_f1=('test_macro_f1', 'std')
    )
)

summary.to_csv(OUT / 'top2_learning_curve_summary.csv', index=False)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.0), constrained_layout=True)

for ax, model, letter in zip(
    axes,
    ['TabPFN', 'LDA_XGBoost'],
    ['A', 'B']
):
    g = summary[summary['model'] == model].sort_values('mean_n_train')

    ax.errorbar(
        g['mean_n_train'],
        g['mean_train_macro_f1'],
        yerr=g['sd_train_macro_f1'],
        marker='o',
        capsize=4,
        linewidth=1.8,
        label='Training macro-F1'
    )
    ax.errorbar(
        g['mean_n_train'],
        g['mean_test_macro_f1'],
        yerr=g['sd_test_macro_f1'],
        marker='o',
        capsize=4,
        linewidth=1.8,
        label='Outer-test macro-F1'
    )

    ymin = min(
        g['mean_test_macro_f1'].min(),
        g['mean_train_macro_f1'].min()
    ) - 0.02
    ax.set_ylim(max(0.80, ymin), 1.005)
    ax.set_xlabel('Training samples')
    ax.set_ylabel('Macro-F1')
    ax.grid(True, alpha=0.18)
    ax.legend(loc='lower right', fontsize=8)
    ax.text(
        0.02, 0.98, letter,
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=13, fontweight='bold'
    )

fig8_png = FIG / 'Figure8_Learning_Curves_TabPFN_LDAXGBoost_PUBLICATION.png'
fig8_pdf = FIG / 'Figure8_Learning_Curves_TabPFN_LDAXGBoost_PUBLICATION.pdf'
fig8_svg = FIG / 'Figure8_Learning_Curves_TabPFN_LDAXGBoost_PUBLICATION.svg'

fig.savefig(fig8_png, dpi=600, bbox_inches='tight')
fig.savefig(fig8_pdf, bbox_inches='tight')
fig.savefig(fig8_svg, bbox_inches='tight')
plt.show()

print('Saved:')
print(fig8_png)
print(fig8_pdf)
print(fig8_svg)


### Figure 8 caption for the manuscript

**Figure 8.** Leakage-safe diagnostic learning curves for the two leading models using the full 11-descriptor representation: **(A)** TabPFN and **(B)** LDA-XGBoost. Curves show mean training and untouched outer-test macro-F1 across the 15 repeated outer-fold evaluations, with error bars representing the standard deviation. Training fractions were sampled only from the corresponding outer-training fold; the outer-test fold remained unchanged at every training size.


## 8. Integrity checks and run metadata


In [ ]:
# Figure 4 integrity
for model in ['TabPFN', 'LDA_XGBoost']:
    assert seed_matrices[model].shape == (3, 4, 4)
    assert np.allclose(seed_matrices[model].sum(axis=2), 1.0)

# Learning-curve integrity
for model in ['TabPFN', 'LDA_XGBoost']:
    g = combined[combined['model'] == model]
    assert len(g) == 75, (model, len(g))
    assert set(np.round(g['train_fraction'], 2)) == set(TRAIN_FRACTIONS)

run_info = {
    'seeds': SEEDS,
    'outer_folds': OUTER_FOLDS,
    'train_fractions': TRAIN_FRACTIONS,
    'feature_set': 'FULL_11',
    'figure4_models': ['TabPFN', 'LDA_XGBoost'],
    'figure4_source': str(OOF_FILE),
    'figure4_aggregation': (
        'Row-normalized confusion matrix computed separately within each seed '
        'from complete outer-test OOF predictions, then averaged across seeds.'
    ),
    'figure8_models': ['TabPFN', 'LDA_XGBoost'],
    'lda_learning_curve_source': str(LC_EXISTING),
    'tabpfn_learning_curve_output': str(final_file),
    'learning_curve_note': (
        'Only outer-training data were subsampled. Outer-test folds remained untouched. '
        'TabPFN was used as the fixed pretrained comparator without fine-tuning.'
    ),
    'device': DEVICE,
}

with open(LOG / 'NB12_run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)

print('✅ NB12 integrity checks passed.')
print('Figure 4 and Figure 8 are ready in:', FIG)
